# Topic Naming with OpenAI

Loads the BERTopic models and their topic info produced in `02-topic_model/`,
sends representative documents per cluster to GPT to obtain:

1. A **description** of each topic cluster.
2. An **enhanced (synthesised) description**.
3. A short **cluster name**.

Prompts are loaded from `prompts.yaml`. Results are saved as TSV files alongside
the topic models in `assets/topic_models/`.

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
from openai import OpenAI

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR     = os.path.join('..', 'assets')
MODELS_DIR     = os.path.join(ASSETS_DIR, 'topic_models')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')

# Number of representative documents to send per cluster
TOP_N_DOCS = 5

# OpenAI model
MODEL = 'gpt-4.1-nano'

# ── Load prompts ──────────────────────────────────────────────────────────────
with open('prompts.yaml') as f:
    prompts = yaml.safe_load(f)

THEME = prompts['theme']
THEME_DESCRIPTION = prompts['theme_description']

print(f'Theme: {THEME}')
print(f'Description: {THEME_DESCRIPTION[:80]}...')

In [ ]:
# ── OpenAI client ─────────────────────────────────────────────────────────────
api_key = open('openai.key').read().strip()
client = OpenAI(api_key=api_key)

## 1. Load topic models and corpora

In [ ]:
# Topic-level info (from BERTopic's get_topic_info())
teams_info  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_topic_info.txt'),  sep='\t')
papers_info = pd.read_csv(os.path.join(MODELS_DIR, 'papers_topic_info.txt'), sep='\t')

# Document-level topic assignments
teams_doc_topics  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_doc_topics.txt'),  sep='\t')
papers_doc_topics = pd.read_csv(os.path.join(MODELS_DIR, 'papers_doc_topics.txt'), sep='\t')

# Full corpora (with text)
teams_corpus  = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'),  sep='\t')
papers_corpus = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t')

# Merge text into doc-topics
teams_df  = teams_doc_topics.merge(teams_corpus,  on='UT', how='left')
papers_df = papers_doc_topics.merge(papers_corpus, on='id', how='left')

print(f'Teams:  {len(teams_df):,} docs, {teams_info.Topic.max() + 1} topics')
print(f'Papers: {len(papers_df):,} docs, {papers_info.Topic.max() + 1} topics')

## 2. Helper functions

In [ ]:
def ask_gpt(
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    temperature: float = 0.1,
    max_tokens: int = 500,
) -> str:
    """Send a system + user prompt to OpenAI and return the assistant reply."""
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
    )
    return response.choices[0].message.content


def get_representative_texts(df: pd.DataFrame, topic_id: int, top_n: int = TOP_N_DOCS) -> str:
    """Return the top-N documents of a topic joined with ##### separators.

    The text is truncated to ~14 000 chars (~3 500 tokens) to stay within
    context limits.
    """
    subset = df[df['topic'] == topic_id]
    texts = subset['text'].head(top_n).tolist()
    bulk = ' ##### '.join(texts)
    return bulk[:14_000]


def fmt_prompt(template: str, **kwargs) -> str:
    """Format a prompt template with the given keyword arguments."""
    result = template
    for key, value in kwargs.items():
        result = result.replace('{' + key + '}', str(value))
    return result

In [ ]:
def name_topics(
    df: pd.DataFrame,
    topic_info: pd.DataFrame,
) -> pd.DataFrame:
    """For each non-outlier topic, get description, enhanced description, and name.

    Returns a DataFrame with columns: topic, description, enhanced_description, name.
    """
    p_desc = prompts['cluster_description']
    p_enh  = prompts['cluster_description_enhanced']
    p_name = prompts['cluster_name']

    rows = []
    topic_ids = sorted(topic_info[topic_info['Topic'] != -1]['Topic'].unique())

    for tid in topic_ids:
        print(f'  Topic {tid} ...', end=' ', flush=True)
        cluster_text = get_representative_texts(df, tid)

        # Step 1: cluster description
        description = ask_gpt(
            system_prompt=fmt_prompt(p_desc['system'], theme=THEME, theme_description=THEME_DESCRIPTION),
            user_prompt=fmt_prompt(p_desc['user'], cluster_text=cluster_text),
            temperature=0.2,
        )

        # Step 2: enhanced (synthesised) description
        enhanced = ask_gpt(
            system_prompt=fmt_prompt(p_enh['system'], theme=THEME),
            user_prompt=fmt_prompt(p_enh['user'], cluster_description=description),
            temperature=0.1,
        )

        # Step 3: short name
        name = ask_gpt(
            system_prompt=fmt_prompt(p_name['system'], theme=THEME, theme_description=THEME_DESCRIPTION),
            user_prompt=fmt_prompt(p_name['user'], cluster_description=description),
            max_tokens=60,
            temperature=0.3,
        )
        name = name.strip().strip('"').strip("'")

        print(name)
        rows.append({
            'topic': tid,
            'name': name,
            'description': enhanced,
            'raw_description': description,
        })

    return pd.DataFrame(rows)

## 3. Name iGEM Teams topics

In [ ]:
print('Naming iGEM Teams topics...')
teams_names = name_topics(teams_df, teams_info)
teams_names

## 4. Name SynBio Papers topics

In [ ]:
print('Naming SynBio Papers topics...')
papers_names = name_topics(papers_df, papers_info)
papers_names

## 5. Save results

In [ ]:
teams_names.to_csv(os.path.join(MODELS_DIR, 'teams_topic_names.txt'), sep='\t', index=False)
papers_names.to_csv(os.path.join(MODELS_DIR, 'papers_topic_names.txt'), sep='\t', index=False)

print(f'Saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')